# 02. CoT 데이터 구조 파악

**목표**: 03 단계에서 품질 판정 함수를 만들기 전에 CoT 데이터의 구조와 형식을 확인한다.
여기서는 **판정하지 않는다.** 판정 규칙을 만들 수 있는지(필요한 정보를 텍스트에서 뽑아낼 수 있는지)만 가늠한다.

| 단계 | 내용 | 관련 결정 |
|---|---|---|
| 1 | 질문유형 × Train/Valid 건수 | D12 |
| 2 | 스키마 점검 (키, 타입, 빈 값, step_count, product_names 이상) | |
| 3 | 텍스트 길이 분포 | |
| 4 | 메타 분포 (category, gender, age) | 06 준비 |
| 5 | 질문유형별 샘플 읽기 | |
| 6 | 품질 기준별 사전 탐색 + 상품명 매칭 실패 원인 분류 | D10 |
| 7 | 보험 상품의 위험등급 정보 확인 | D12 |

데이터를 읽는 함수는 `src/load_cot.py`에 있다.

In [ ]:
# 라이브러리와 프로젝트 함수 불러오기
import re
import sys
from pathlib import Path

import pandas as pd

# 노트북은 notebooks/ 안에 있으므로, 한 단계 위(프로젝트 루트)를 import 경로에 추가
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load_cot import load_cot, load_products

# 표 출력 옵션
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

In [ ]:
# CoT 전체(Train + Valid)와 상품 원천데이터(Training 폴더) 읽기
# 상품 원천데이터는 Training과 Validation이 동일하므로 하나만 읽는다 (D1)
cot = load_cot()
prod = load_products()
print("CoT:", cot.shape, "/ 상품:", prod.shape)

## 1. 질문유형 × Train/Valid 건수

In [ ]:
# 질문유형(폴더 기준)별로 Train/Valid 건수를 센다
counts = pd.crosstab(cot["query_group"], cot["split"], margins=True, margins_name="합계")
counts["Valid 비율(%)"] = (counts["Validation"] / counts["합계"] * 100).round(1)
counts

In [ ]:
# 폴더 기준 질문유형과 파일 안의 query_type·category 값이 서로 일치하는지 확인
pd.crosstab(cot["query_group"], [cot["category"], cot["query_type"]]).T

## 2. 스키마 점검

In [ ]:
# 원래 JSON 키 12개의 자료형 구성 확인
# (파이썬 타입 이름별로 몇 건인지 센다. 한 컬럼에 타입이 섞여 있으면 여러 줄이 나온다)
KEYS = ["cot_id", "category", "gender", "age", "query_type", "question",
        "cot1", "cot2", "cot3", "answer", "step_count", "product_names_raw"]
type_tbl = pd.DataFrame({k: cot[k].map(lambda v: type(v).__name__).value_counts() for k in KEYS}).T.fillna(0).astype(int)
type_tbl

In [ ]:
# 텍스트 컬럼의 빈 값(None, 빈 문자열, 공백만 있는 문자열) 개수
TEXT = ["question", "cot1", "cot2", "cot3", "answer"]
empty = {c: cot[c].isna().sum() + (cot[c].astype(str).str.strip() == "").sum() for c in TEXT}
pd.Series(empty, name="빈 값 개수")

In [ ]:
# cot_id 중복 여부 (Train과 Valid 사이 포함)
print("cot_id 고유값 수:", cot["cot_id"].nunique(), "/ 전체 행:", len(cot))
dup = cot[cot["cot_id"].duplicated(keep=False)]
print("중복 cot_id 행 수:", len(dup))

# step_count 분포
print(cot["step_count"].value_counts().sort_index())

In [ ]:
# product_names를 리스트로 바꾸지 못한 건 (깨진 문자열)
# 원문 전체를 출력하지 않고 길이와 앞뒤 일부만 본다
bad = cot.loc[~cot["product_names_ok"], ["split", "query_group", "cot_id", "product_names_raw"]].copy()
bad["길이"] = bad["product_names_raw"].str.len()
bad["앞 40자"] = bad["product_names_raw"].str[:40]
bad.drop(columns="product_names_raw")

In [ ]:
# 추천 상품 개수 분포 (질문유형별)
cot["n_products"] = cot["product_names"].map(len)
pd.crosstab(cot["query_group"], cot["n_products"])

## 3. 텍스트 길이 분포

In [ ]:
# 컬럼별 글자 수 요약 통계
lengths = pd.DataFrame({c: cot[c].astype(str).str.len() for c in TEXT})
lengths.describe().round(0)

In [ ]:
# 질문유형별 글자 수 중앙값 (유형마다 길이가 다른지)
lengths.assign(query_group=cot["query_group"]).groupby("query_group").median().round(0)

In [ ]:
# 컬럼별로 가장 짧은 3건의 cot_id와 길이 (극단치 확인용, 본문은 출력하지 않음)
for c in TEXT:
    idx = lengths[c].nsmallest(3).index
    print(c, list(zip(cot.loc[idx, "cot_id"], lengths.loc[idx, c])))

## 4. 메타 분포 (category, gender, age)

In [ ]:
# 성별, 연령대 분포 (category별)
for col in ["gender", "age"]:
    print(f"=== {col} ===")
    print(pd.crosstab(cot[col], cot["category"], margins=True, margins_name="합계"))
    print()

## 5. 질문유형별 샘플 1건 읽기

각 필드는 앞 250자만 출력한다. 전체 흐름(question → cot1 → cot2 → cot3 → answer)을 보는 용도다.

In [ ]:
# 질문유형마다 cot_id가 가장 작은 1건을 골라 앞부분만 출력
for g, sub in cot.groupby("query_group"):
    r = sub.sort_values("cot_id").iloc[0]
    print("=" * 100)
    print(f"[{g}] cot_id={r['cot_id']} / {r['gender']} {r['age']} / 추천 {r['n_products']}개")
    for c in TEXT:
        print(f"--- {c} ---")
        print(str(r[c])[:250].replace("\n", " "), "...")

## 6. 품질 기준별 사전 탐색

**주의**: 아래 키워드 목록은 **임시 후보**다. 판정 규칙이 아니다.
"이 단어가 등장하는 비율"만 보고, 03 단계에서 규칙을 만들 때 참고한다.
키워드가 있다고 해서 그 정보가 충족된 것은 아니고, 없다고 해서 누락된 것도 아니다.

### 6-1. 기준 1: 6개 필수확인정보 관련 표현 (question + cot1)

In [ ]:
# 6개 필수확인정보별 임시 키워드 후보 (정규식)
INFO_KEYWORDS = {
    "거래목적":   r"목적|목표|마련|대비|위해",
    "재산상황":   r"자산|소득|재산|여유자금|\d+\s*(?:만|억)\s*원",
    "투자경험":   r"경험|투자해|운용해|가입해",
    "상품이해도": r"이해|잘 모르|익숙|지식",
    "위험태도":   r"위험|손실|원금|안정|공격|변동",
    "연령":       r"\d0대|\d+세|중반|초반|후반",
}

# question과 cot1을 합친 텍스트에서 키워드가 하나라도 있으면 True
qc = cot["question"].astype(str) + " " + cot["cot1"].astype(str)
hit = pd.DataFrame({k: qc.str.contains(p, regex=True) for k, p in INFO_KEYWORDS.items()})

# 질문유형별 등장 비율(%)
(hit.groupby(cot["query_group"]).mean() * 100).round(1)

### 6-2. 기준 2: cot1의 투자성향 표현

In [ ]:
# 투자성향 5단계 명칭이 cot1에 몇 종류 등장하는지
# '안정형'과 '안정추구형'은 글자가 겹치지 않으므로 그대로 찾는다
PROPENSITY = ["공격투자형", "적극투자형", "위험중립형", "안정추구형", "안정형"]
found = cot["cot1"].astype(str).apply(lambda t: [p for p in PROPENSITY if p in t])
cot["n_propensity_cot1"] = found.map(len)
pd.crosstab(cot["query_group"], cot["n_propensity_cot1"])

In [ ]:
# 성향 명칭 대신 쓰는 표현이 있는지 보기 위해, 명칭이 0개인 cot1에서 '형' 앞 단어 상위 빈도 확인
no_prop = cot.loc[cot["n_propensity_cot1"] == 0, "cot1"].astype(str)
words = no_prop.str.findall(r"(\S{2,6}형)").explode().value_counts()
print("명칭 0개인 cot1 수:", len(no_prop))
words.head(15)

### 6-3. 기준 3: cot2 허용 위험등급 추출 가능성 + 상품 위험등급 형식

In [ ]:
# cot2에서 'N등급' 표현을 모두 뽑는다
cot["cot2_grades"] = cot["cot2"].astype(str).str.findall(r"([1-6])\s*등급").map(lambda x: sorted(set(map(int, x))))
cot["cot2_has_grade"] = cot["cot2_grades"].map(len) > 0
(pd.crosstab(cot["query_group"], cot["cot2_has_grade"], normalize="index") * 100).round(1)

In [ ]:
# 증권 상품 JSON의 risk_grade 값 형식 (상위 값)
prod.loc[prod["domain"] == "증권", "risk_grade"].value_counts().head(10)

### 6-4. 상품명 매칭 실패 원인 분류 (D10)

CoT `product_names`의 각 상품명을 상품 JSON과 아래 순서로 대조한다. 앞 단계에서 맞으면 그 원인으로 분류한다.

| 순서 | 분류 | 비교 방법 |
|---|---|---|
| 0 | 정확 일치 | `product_name`과 완전히 같음 |
| 1 | 보험 전체명 일치 | 보험 상품의 `product_full_name`과 같음 |
| 2 | 보험 모상품명 일치 | 보험 상품의 `mother_product_name`과 같음 |
| 3 | 공백 차이 | 공백을 모두 지우면 같음 |
| 4 | 공백·괄호·기호 차이 | 공백, 괄호, 대괄호, 하이픈, 따옴표를 지우면 같음 |
| 5 | 부분 일치 | 정규화한 이름이 다른 이름에 포함되거나 포함함 (후보 1개 이상) |
| 6 | 일치 없음 | 위 어느 것에도 해당 없음 |

애매한 경우 처리: 순서가 앞인 분류를 우선한다. 5(부분 일치)는 후보가 여러 개일 수 있으며, 같은 상품이라고 확정하지 않는다.

In [ ]:
# 비교용 정규화 함수
def norm_space(s):
    # 공백만 제거
    return re.sub(r"\s+", "", str(s))

def norm_full(s):
    # 공백, 괄호류, 하이픈, 따옴표, 가운뎃점 제거
    return re.sub(r"[\s\(\)\[\]\{\}\-_'\"·,.]", "", str(s))

# 상품 JSON 쪽 이름 집합 준비
names_exact = set(prod["product_name"].dropna())
ins = prod[prod["domain"] == "보험"]
names_full = set(ins["product_full_name"].dropna())
names_mother = set(ins["mother_product_name"].dropna())
all_names = pd.concat([prod["product_name"], ins["product_full_name"], ins["mother_product_name"]]).dropna().unique()
space_set = {norm_space(n) for n in all_names}
full_list = [norm_full(n) for n in all_names]
full_set = set(full_list)

def classify(name):
    if name in names_exact: return "0 정확 일치"
    if name in names_full: return "1 보험 전체명 일치"
    if name in names_mother: return "2 보험 모상품명 일치"
    if norm_space(name) in space_set: return "3 공백 차이"
    n = norm_full(name)
    if n in full_set: return "4 공백·괄호·기호 차이"
    if len(n) >= 4 and any((n in m) or (m in n and len(m) >= 4) for m in full_list): return "5 부분 일치"
    return "6 일치 없음"

# CoT 한 건에 상품이 여러 개이므로 상품명 단위로 펼친다
rec = cot[["split", "query_group", "category", "cot_id", "product_names"]].explode("product_names")
rec = rec.dropna(subset=["product_names"]).rename(columns={"product_names": "rec_name"})
rec["match_cause"] = rec["rec_name"].map(classify)
print("추천 상품명 수(중복 포함):", len(rec))
rec["match_cause"].value_counts().sort_index().to_frame("건수").assign(**{"비율(%)": lambda d: (d["건수"] / d["건수"].sum() * 100).round(1)})

In [ ]:
# 업종(category)별 원인 분포 (%)
(pd.crosstab(rec["category"], rec["match_cause"], normalize="index") * 100).round(1)

In [ ]:
# 원인별 예시 5건 (정확 일치 제외). 추천 상품명과, 공백·괄호 제거 후 가장 비슷한 상품 JSON 이름을 나란히 본다
import difflib
for cause, sub in rec[rec["match_cause"] != "0 정확 일치"].groupby("match_cause"):
    print("=" * 90)
    print(cause, f"({len(sub)}건)")
    for nm in sub["rec_name"].drop_duplicates().head(5):
        close = difflib.get_close_matches(norm_full(nm), full_list, n=1, cutoff=0.0)
        print("  추천:", nm)
        print("  유사:", close[0] if close else "-")

#### 6-4-1. '부분 일치'와 '일치 없음'의 세부 원인

위 예시에서 두 가지 패턴이 보여 따로 센다.
- **'본인' → '고객' 치환 의심**: 추천명의 '고객'을 '본인'으로 되돌리면 상품 JSON 이름과 일치하는 경우 (예: 일고객덱스 ↔ 일본인덱스). 데이터 생성 과정의 텍스트 치환으로 보이지만, **원인은 확인되지 않았다.**
- **상품명 조각**: 상품명 하나가 쉼표 등으로 잘려 리스트의 여러 원소로 들어간 것으로 보이는 경우. 판별 기준(임시): 정규화 후 10자 이하이거나 '['로 시작.

애매한 경우 처리: 두 패턴에 모두 해당하면 '본인→고객 치환'을 우선한다. 어느 쪽도 아니면 '기타'로 남긴다.

In [ ]:
# 부분 일치(5)와 일치 없음(6)만 골라 세부 원인을 붙인다
def sub_cause(name):
    restored = norm_full(str(name).replace("고객", "본인"))
    if restored in full_set:
        return "본인→고객 치환 의심"
    n = norm_full(name)
    if len(n) <= 10 or str(name).strip().startswith("["):
        return "상품명 조각"
    return "기타"

tail = rec[rec["match_cause"].isin(["5 부분 일치", "6 일치 없음"])].copy()
tail["sub_cause"] = tail["rec_name"].map(sub_cause)
print(pd.crosstab(tail["match_cause"], [tail["category"], tail["sub_cause"]]))

# '기타'로 남은 이름은 전부 보여 준다 (건수가 적음)
print("\n[기타] 추천명 목록")
for nm in tail.loc[tail["sub_cause"] == "기타", "rec_name"].drop_duplicates():
    print(" -", nm)

### 6-5. 기준 4: answer의 위험 설명 표현

In [ ]:
# answer에 손실·원금·위험등급 관련 표현이 있는지 (임시 키워드)
RISK_KEYWORDS = {
    "손실":      r"손실",
    "원금":      r"원금",
    "N등급":     r"[1-6]\s*등급",
    "위험(단어)": r"위험",
}
ans = cot["answer"].astype(str)
risk_hit = pd.DataFrame({k: ans.str.contains(p, regex=True) for k, p in RISK_KEYWORDS.items()})
(risk_hit.groupby(cot["query_group"]).mean() * 100).round(1)

## 7. 보험 상품의 위험등급 정보 (D12)

In [ ]:
# 보험 상품 JSON에 risk_grade 컬럼 값이 있는지
print("보험 상품 수:", len(ins))
print("risk_grade 값이 있는 보험 상품:", ins["risk_grade"].notna().sum() if "risk_grade" in ins else "컬럼 없음")

# 보험 상품 JSON의 컬럼 중 '등급'·'위험'과 관련된 값이 들어 있는 컬럼 찾기
# pandas 3부터 문자열 컬럼은 object가 아니라 str 타입이므로 is_string_dtype/object 둘 다 확인
ins_text_cols = [c for c in ins.columns
                 if pd.api.types.is_string_dtype(ins[c]) or ins[c].dtype == object]
mention = {c: ins[c].astype(str).str.contains(r"등급|위험", regex=True).sum() for c in ins_text_cols}
pd.Series(mention, name="'등급/위험' 포함 상품 수").loc[lambda s: s > 0].sort_values(ascending=False)

In [ ]:
# 보험 상품 유형(product_type)과 원천데이터 분류(product_group) 대응
pd.crosstab(ins["product_group"], ins["product_type"])

In [ ]:
# 보험 CoT의 질문유형별로 cot2에 'N등급' 표현이 있는 비율과, 추천 상품이 어느 보험 분류로 매칭되는지
ins_cot = cot[cot["category"] == "보험"]
print((ins_cot.groupby("query_group")["cot2_has_grade"].mean() * 100).round(1))

# 추천 상품명을 보험 product_full_name 또는 mother_product_name으로 연결해 product_group을 붙인다
# 애매한 경우: 같은 이름이 여러 분류에 있으면 분류를 '/'로 이어 붙여 표시한다
name2group = (pd.concat([ins[["product_full_name", "product_group"]].rename(columns={"product_full_name": "n"}),
                         ins[["mother_product_name", "product_group"]].rename(columns={"mother_product_name": "n"}),
                         ins[["product_name", "product_group"]].rename(columns={"product_name": "n"})])
              .dropna().drop_duplicates().groupby("n")["product_group"].apply(lambda s: "/".join(sorted(set(s)))))
ins_rec = rec[rec["category"] == "보험"].copy()
ins_rec["matched_group"] = ins_rec["rec_name"].map(name2group).fillna("(매칭 안 됨)")
pd.crosstab(ins_rec["query_group"], ins_rec["matched_group"])

## 02 정리 (직접 작성)

1. 질문유형별 건수와 Train/Valid 비율을 보고, D12에서 Train/Valid를 어떻게 다룰 것인가?
2. 기준 1~4 각각, 텍스트에서 판정에 필요한 정보를 뽑아낼 수 있어 보이는가? 어려운 기준은 무엇이고 왜인가?
3. D10: 매칭 실패 원인 분포를 보고 어떤 매칭 규칙을 쓸 것인가?
4. D12: 보험 상품에 위험등급이 있는가? 보험 CoT에 기준 3을 적용할 수 있는가? 진단 범위를 어떻게 확정할 것인가?
5. 03 판정 함수에서 애매한 경우로 따로 처리해야 할 것:
